        # 02 Random と Greedy

        - **Random Matching**：相互承諾可能な組をランダムな順に調べ、2 人とも空いていれば成立
        - **Greedy Matching**：Group A を順番に処理し、空いていて相互承諾可能な相手のうち希望順位が最も高い人と成立（変更しない）
        - 比べる指標：成立率、初期満足度、試行間のばらつき（RQ1）

> **教育用の架空データ・架空モデル**です。参加者は実在の人物ではなく、Group A・B はアルゴリズム上の区分で性別や性的指向を意味しません。希望スコアは実際の結婚適性を表さず、関係継続モデルの係数は実際の離婚率や実証された因果効果ではありません。

## 準備：実験環境（最初に必ず実行）

F3 リポジトリの `.py`（計算の本体）を import できるように、作業ディレクトリ（F3 のルート）を自動で探して設定します。

| 実行場所 | 準備 |
| --- | --- |
| ローカルの Jupyter / VSCode | `notebooks/` の中で開けば、そのまま動きます |
| Colab（方法A：GitHub から clone） | 下のセルの `REPO_URL` に**自分たちのリポジトリの URL** を入れる |
| Colab（方法B：ZIP をアップロード） | F3 フォルダを ZIP にして `/content` にアップロードしておく（またはセルの実行中にアップロード） |

Colab の CPU ランタイムで動きます（GPU・TPU は不要）。`textbook/`（教科書のコード）は公開リポジトリに含めない約束です。`textbook/` がないと import が失敗します。

In [ ]:
#@title 準備（F3 の .py を読み込む）
import os, sys, glob, zipfile, subprocess
from pathlib import Path

# ---- 方法A：GitHub から clone する場合だけ設定する（URL は自分たちのものを入れる）----
REPO_URL = ""                              # 例："https://github.com/<team>/<repo>.git"（空なら clone しない）
REPO_SUBDIR = "f3-matching-algorithms"     # リポジトリの中の F3 フォルダ（ルートが F3 なら ""）

IN_COLAB = "google.colab" in sys.modules


def is_f3_root(p):
    return (Path(p) / "matching_algorithms.py").exists() and (Path(p) / "relationship_stability.py").exists()


def find_f3_root():
    for p in [Path.cwd(), Path.cwd().parent]:          # ローカル：notebooks/ の中、または F3 のルート
        if is_f3_root(p):
            return p.resolve()
    if not IN_COLAB:
        return None
    if REPO_URL:                                        # Colab 方法A
        dest = Path("/content/repo")
        if not dest.exists():
            subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(dest)], check=True)
        cand = dest / REPO_SUBDIR if REPO_SUBDIR else dest
        if is_f3_root(cand):
            return cand
    zips = glob.glob("/content/*.zip")                  # Colab 方法B
    if not zips:
        from google.colab import files
        print("F3 フォルダの ZIP を選んでアップロードしてください")
        zips = ["/content/" + name for name in files.upload()]
    for z in zips:
        with zipfile.ZipFile(z) as zf:
            zf.extractall("/content/f3_upload")
    for hit in glob.glob("/content/f3_upload/**/matching_algorithms.py", recursive=True):
        if is_f3_root(Path(hit).parent):
            return Path(hit).parent
    return None


F3_ROOT = find_f3_root()
if F3_ROOT is None:
    raise RuntimeError("F3 のフォルダが見つかりません。上の説明（方法A／方法B）を確認してください。")
os.chdir(F3_ROOT)
if str(F3_ROOT) not in sys.path:
    sys.path.insert(0, str(F3_ROOT))
if not (F3_ROOT / "textbook" / "fixed_queue.py").exists():
    print("⚠️ textbook/ がありません。preferences.py・matching_algorithms.py の import が失敗します。")

# ---- 既存の .py（計算の本体）と、グラフ ----
import matplotlib.pyplot as plt
import participants as pt
import preferences as pr
import matching_algorithms as ma
import relationship_stability as rs
import evaluation as ev
import matching_charts as mc

mc.setup_style()                       # 文字を大きくするなら mc.setup_style(1.3)
groups = pt.load_participants()
scores = pr.score_table(groups)
rankings = pr.build_rankings(groups, scores)
params = rs.load_parameters()          # 関係継続モデルのパラメータ（すべて架空）
A_IDS = [p["id"] for p in groups["A"]]
B_IDS = [p["id"] for p in groups["B"]]
N_TOTAL = pt.total_participants(groups)
compat = lambda a, b: pr.pair_compatibility(a, b, scores)

SAVE_FIGURES = False                   # True にすると、グラフを notebooks/figures/ に PNG で保存する
FIG_DIR = "notebooks/figures"


def show(fig, name):
    """グラフを表示し、SAVE_FIGURES = True なら PNG も保存する"""
    if SAVE_FIGURES:
        print("saved:", mc.save(fig, f"{FIG_DIR}/{name}.png"))
    plt.show()


def table(rows, columns):
    """辞書のリストを表で表示する。columns = [(キー, 見出し, 書式)]。pandas があれば DataFrame で表示"""
    data = [{h: ("-" if r.get(k) is None else f.format(r[k])) for k, h, f in columns} for r in rows]
    try:
        import pandas as pd
        from IPython.display import display
        display(pd.DataFrame(data))
    except ImportError:
        heads = [h for _k, h, _f in columns]
        widths = [max(len(h), *(len(d[h]) for d in data)) for h in heads]
        print("  ".join(h.rjust(w) for h, w in zip(heads, widths)))
        for d in data:
            print("  ".join(d[h].rjust(w) for h, w in zip(heads, widths)))


print("F3_ROOT:", F3_ROOT)
print(f"参加者 Group A {len(A_IDS)} 人・Group B {len(B_IDS)} 人 / 関係継続モデル（架空）:",
      {k: params[k] for k in ("baseline_hazard", "alpha", "beta", "gamma", "default_annual_stress",
                               "default_conflict_resolution", "simulation_years", "random_seed")})

## 評価指標

- **成立率** = 2 × 成立ペア数 / 全参加者数
- **初期満足度**：受け入れ可能な相手が L 人いて、k 番目の相手とマッチング → $1 - (k-1)/L$（第 1 希望 = 1）。**未マッチングは 0**。平均は全参加者で取る

In [ ]:
for name, m in [("random (seed 0)", ma.random_matching(rankings, A_IDS, B_IDS, seed=0)),
                ("greedy (ID order)", ma.greedy_matching(rankings, A_IDS, B_IDS))]:
    s = ev.satisfaction_summary(m, rankings, A_IDS, B_IDS)
    print(f"{name:18}: {dict(sorted(m.items()))}  成立率 {ev.matching_rate(m, N_TOTAL):.0%}  満足度 {s['mean_all']:.2f}")

## Greedy の処理順序

Greedy は**先に処理された人が有利**です。処理順序（`greedy_order(seed)`）を変えると結果が変わることがあります。

In [ ]:
for seed in [None, 1, 2, 3]:
    order = ma.greedy_order(A_IDS, seed)
    m = ma.greedy_matching(rankings, A_IDS, B_IDS, order)
    s = ev.satisfaction_summary(m, rankings, A_IDS, B_IDS)
    print(f"order {' '.join(order)}: {dict(sorted(m.items()))}  満足度 {s['mean_all']:.2f}  A {s['mean_A']:.2f} / B {s['mean_B']:.2f}")

## 複数シードでの反復

Random はシードごとに、Greedy は処理順序のシードごとに実行して、ばらつきを見ます（**1 回の結果だけで判断しない**）。

In [ ]:
#@title グラフを表示（コードは隠しています）
SEEDS = range(50)                       # TODO：試行回数を変えてみよう
trials = {"random": [], "greedy": []}
sat = {"random": [], "greedy": []}
for s in SEEDS:
    for name, m in [("random", ma.random_matching(rankings, A_IDS, B_IDS, seed=s)),
                    ("greedy", ma.greedy_matching(rankings, A_IDS, B_IDS, ma.greedy_order(A_IDS, s)))]:
        trials[name].append(ev.matching_rate(m, N_TOTAL))
        sat[name].append(ev.satisfaction_summary(m, rankings, A_IDS, B_IDS)["mean_all"])
summary = {n: {"matching_rate": sum(trials[n]) / len(SEEDS), "satisfaction": sum(sat[n]) / len(SEEDS)} for n in trials}
fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
mc.plot_metric_bars(summary, "matching_rate", f"Matching rate (mean of {len(SEEDS)} seeds)", "matching rate", ax=axes[0], ylim=(0, 1.1))
mc.plot_metric_bars(summary, "satisfaction", f"Initial satisfaction (mean of {len(SEEDS)} seeds)", "satisfaction",
                    fmt="{:.2f}", ax=axes[1], ylim=(0, 1.1))
fig.tight_layout()
show(fig, "02_bars")

In [ ]:
#@title グラフを表示（コードは隠しています）
show(mc.plot_trials(sat, f"Initial satisfaction across {len(SEEDS)} seeds (Greedy: random processing order)",
                    "mean satisfaction"), "02_trials")

## 学生向け TODO・考察

- [ ] 試行回数 `SEEDS` を変え、平均が安定するか調べる
- [ ] Greedy の処理順序で、誰の満足度が上がり、誰が下がるかを調べる（公平性）
- [ ] 参加者を増やした・希望条件を厳しくした場合、Random の成立率は Greedy より低くなるか（RQ1・RQ2）

**考察（チームで記入）**：TODO

## 参考文献

- D. Gale, L. S. Shapley, "College Admissions and the Stability of Marriage", *The American Mathematical Monthly*, 69(1), 9–15, 1962.
- 柴田望洋『新・明解 Pythonで学ぶアルゴリズムとデータ構造（第2版）』SBクリエイティブ（第3章 線形探索・ハッシュ法、第4章 キュー、第6章 挿入ソート）
- F3 リポジトリの `README.md`（モデルの仮定・計算量・モデルの限界と倫理的配慮）
- データ：`data/sample_participants.json`、`data/relationship_parameters.json`（この教材のための**架空のデータ**）